# The Orthodox Canonical Form (OCF)

The Orthodox Canonical Form defines four member functions that every class managing resources **MUST** implement to behave correctly. Without them, C++ will generate default versions that may not do what you intend.

These four functions are:
1. Default constructor
2. Copy constructor
3. Copy assignment operator
4. Destructor

If your class manages a resource (heap memory, file handles, sockets), neglecting any one of these can lead to crashes, memory leaks, or silent data corruption.

## Why OCF Exists

When a class holds a raw pointer to heap memory, C++ default behavior is **shallow copy** — it copies the pointer value, not the data it points to.

```
Before copy:
  bufA.data --> [ 'H','e','l','l','o','\0' ]

After shallow copy (bufB = bufA):
  bufA.data -->\
                [ 'H','e','l','l','o','\0' ]   <-- SAME MEMORY
  bufB.data -->/

When bufA is destroyed:
  delete bufA.data  --> memory freed
  bufB.data --> DANGLING POINTER (undefined behaviour!)
```

The fix is a **deep copy**: allocate fresh memory and copy the contents, so each object owns its own data.

In [ ]:
// Demonstration: what happens with the compiler-generated (shallow) copy
#include <iostream>
#include <cstring>

class Buffer {
public:
    char *data;
    int   size;

    Buffer(const char *str) {
        size = strlen(str) + 1;
        data = new char[size];
        strcpy(data, str);
    }

    // No destructor, no copy constructor -- compiler generates shallow versions
};

{
    Buffer bufA("Hello");
    Buffer bufB = bufA;           // shallow copy: bufB.data == bufA.data

    std::cout << "bufA.data address: " << (void*)bufA.data << std::endl;
    std::cout << "bufB.data address: " << (void*)bufB.data << std::endl;
    std::cout << "Same pointer? " << (bufA.data == bufB.data ? "YES (BUG!)" : "No") << std::endl;
    // When both go out of scope, delete[] is called TWICE on the same pointer -> crash
}

## The Four OCF Functions

| Function | Signature | When Called |
|---|---|---|
| Default constructor | `ClassName()` | Object created with no arguments |
| Copy constructor | `ClassName(const ClassName &other)` | Object initialised from another object |
| Copy assignment operator | `ClassName &operator=(const ClassName &other)` | Existing object assigned from another (`a = b`) |
| Destructor | `~ClassName()` | Object goes out of scope or is `delete`d |

For the rest of this notebook we will build `StringBox`, a class that holds a dynamically allocated C-string.

## 1. Default Constructor

```cpp
ClassName();
```

The default constructor must leave the object in a **valid, usable state** — even if that state is "empty". For a class holding a pointer, initialise it to `NULL` (not garbage) so the destructor can safely `delete` it.

In [ ]:
#include <iostream>
#include <cstring>

class StringBox {
public:
    // Default constructor: empty box
    StringBox() : _content(NULL), _length(0) {
        std::cout << "[StringBox] Default constructor called" << std::endl;
    }

    // Convenience constructor: initialise with a C-string
    StringBox(const char *str) {
        std::cout << "[StringBox] Param constructor called" << std::endl;
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    void print() const {
        std::cout << "StringBox contains: "
                  << (_content ? _content : "<empty>") << std::endl;
    }

private:
    char        *_content;
    unsigned int _length;
};

StringBox emptyBox;
emptyBox.print();

StringBox helloBox("Hello, 42!");
helloBox.print();

## 2. Destructor

```cpp
~ClassName();
```

The destructor **frees every resource** owned by the object. For every `new` there must be exactly one `delete`. For every `new[]` there must be exactly one `delete[]`.

A good practice: set the pointer to `NULL` after deleting so that if the destructor were somehow called twice (it shouldn't be, but defensively), it won't crash.

In [ ]:
// We will rebuild StringBox step by step.
// This cell focuses on the destructor.
// (In the notebook the full class will be assembled in section 8.)

#include <iostream>
#include <cstring>

class StringBoxV2 {
public:
    StringBoxV2() : _content(NULL), _length(0) {}

    StringBoxV2(const char *str) {
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    // Destructor
    ~StringBoxV2() {
        std::cout << "[~StringBoxV2] Freeing: "
                  << (_content ? _content : "<empty>") << std::endl;
        delete[] _content;   // safe even if _content is NULL
        _content = NULL;
    }

    void print() const {
        std::cout << (_content ? _content : "<empty>") << std::endl;
    }

private:
    char        *_content;
    unsigned int _length;
};

{
    StringBoxV2 sb("Destructor demo");
    sb.print();
}  // <-- destructor called automatically here

std::cout << "After block: sb has been destroyed" << std::endl;

## 3. Copy Constructor

```cpp
ClassName(const ClassName &other);
```

Called when a **new** object is initialised from an existing one:
- `StringBox b = a;`
- `StringBox b(a);`
- Passing by value to a function
- Returning by value from a function

The copy constructor must **deep copy**: allocate new memory and copy the data, so the two objects are independent.

In [ ]:
#include <iostream>
#include <cstring>

class StringBoxV3 {
public:
    StringBoxV3() : _content(NULL), _length(0) {}

    StringBoxV3(const char *str) {
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    // Copy constructor -- deep copy
    StringBoxV3(const StringBoxV3 &other) {
        std::cout << "[Copy constructor] copying from: "
                  << (other._content ? other._content : "<empty>") << std::endl;
        if (other._content) {
            _length  = other._length;
            _content = new char[_length + 1];
            strcpy(_content, other._content);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    ~StringBoxV3() { delete[] _content; }

    void print() const {
        std::cout << "  address=" << (void*)_content
                  << " value=[" << (_content ? _content : "<empty>") << "]" << std::endl;
    }

    // We will add setContent and operator= in later cells

private:
    char        *_content;
    unsigned int _length;
};

StringBoxV3 original("Deep copy!");
StringBoxV3 copyObj(original);   // copy constructor

std::cout << "original: "; original.print();
std::cout << "copy:     "; copyObj.print();
std::cout << "Pointers are different: "
          << "(see different addresses above)" << std::endl;

**Exercise 1:** Add a `setContent(const char* str)` method to `StringBoxV3` (or a new version of the class). Create two `StringBox` objects, copy one to another using the copy constructor, then modify the original using `setContent`. Verify the copy was unaffected by printing both.

In [ ]:
// Your code here

## 4. Copy Assignment Operator

```cpp
ClassName &operator=(const ClassName &other);
```

Called when **assigning to an already-constructed object**:
```cpp
StringBox a("first");
StringBox b("second");
b = a;   // assignment operator -- b already exists
```

Key differences from the copy constructor:
- The object **already has data** — you must free the old data first.
- You must handle **self-assignment** (`a = a`) safely.
- You must return `*this` to allow chaining (`a = b = c`).

### Steps:
1. Check for self-assignment: `if (this == &other) return *this;`
2. Free existing resources.
3. Deep copy from `other`.
4. Return `*this`.

In [ ]:
#include <iostream>
#include <cstring>

class StringBoxV4 {
public:
    StringBoxV4() : _content(NULL), _length(0) {}

    StringBoxV4(const char *str) {
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    StringBoxV4(const StringBoxV4 &other) {
        if (other._content) {
            _length  = other._length;
            _content = new char[_length + 1];
            strcpy(_content, other._content);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }

    // Copy assignment operator
    StringBoxV4 &operator=(const StringBoxV4 &other) {
        std::cout << "[operator=] called" << std::endl;

        // Step 1: self-assignment guard
        if (this == &other)
            return *this;

        // Step 2: free existing data
        delete[] _content;

        // Step 3: deep copy
        if (other._content) {
            _length  = other._length;
            _content = new char[_length + 1];
            strcpy(_content, other._content);
        } else {
            _content = NULL;
            _length  = 0;
        }

        // Step 4: return this object
        return *this;
    }

    ~StringBoxV4() { delete[] _content; }

    void print() const {
        std::cout << (_content ? _content : "<empty>") << std::endl;
    }

private:
    char        *_content;
    unsigned int _length;
};

StringBoxV4 a("Alpha");
StringBoxV4 b("Beta");

std::cout << "Before assignment: a="; a.print();
std::cout << "Before assignment: b="; b.print();

b = a;    // assignment operator

std::cout << "After  assignment: a="; a.print();
std::cout << "After  assignment: b="; b.print();

// Self-assignment (should be safe)
a = a;
std::cout << "After self-assign: a="; a.print();

## Putting It All Together

Below is the complete, correct OCF for `StringBox`. Study how all four functions work together.

In [ ]:
#include <iostream>
#include <cstring>

class StringBox {
public:
    // 1. Default constructor
    StringBox() : _content(NULL), _length(0) {
        std::cout << "[StringBox] default ctor" << std::endl;
    }

    // Convenience constructor
    StringBox(const char *str) {
        std::cout << "[StringBox] param ctor: " << (str ? str : "NULL") << std::endl;
        _copyFrom(str);
    }

    // 2. Copy constructor
    StringBox(const StringBox &other) {
        std::cout << "[StringBox] copy ctor" << std::endl;
        _copyFrom(other._content);
    }

    // 3. Copy assignment operator
    StringBox &operator=(const StringBox &other) {
        std::cout << "[StringBox] operator=" << std::endl;
        if (this == &other)
            return *this;
        delete[] _content;
        _copyFrom(other._content);
        return *this;
    }

    // 4. Destructor
    ~StringBox() {
        std::cout << "[StringBox] dtor: " << (_content ? _content : "<empty>") << std::endl;
        delete[] _content;
        _content = NULL;
    }

    void setContent(const char *str) {
        delete[] _content;
        _copyFrom(str);
    }

    void print() const {
        std::cout << "StringBox[" << (_content ? _content : "<empty>") << "]" << std::endl;
    }

private:
    char        *_content;
    unsigned int _length;

    void _copyFrom(const char *str) {
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        } else {
            _content = NULL;
            _length  = 0;
        }
    }
};

In [ ]:
// Test all four OCF functions
std::cout << "--- Creating objects ---" << std::endl;
StringBox *p1 = new StringBox("First");
StringBox *p2 = new StringBox(*p1);    // copy constructor
StringBox *p3 = new StringBox();

std::cout << "\n--- Assigning ---" << std::endl;
*p3 = *p1;    // assignment operator

std::cout << "\n--- Modifying original ---" << std::endl;
p1->setContent("Modified");
p1->print();
p2->print();   // should still be "First"
p3->print();   // should still be "First"

std::cout << "\n--- Destroying ---" << std::endl;
delete p1;
delete p2;
delete p3;

**Exercise 2:** Implement the full OCF for a class `DynamicArray` that holds an `int*` array and a `size` member. The copy constructor and copy assignment operator must deep-copy the array. Include a constructor `DynamicArray(int size)` that allocates the array and zeroes it, and a `set(int index, int value)` / `get(int index)` pair.

In [ ]:
// Your code here

## Canonical vs Non-Canonical

### When can you skip OCF?

Only if your class **does not own any resources** — no raw pointers that it `new`s and must `delete`, no file handles, etc. Examples:
- A class with only `int`, `double`, or other value-type members.
- A class whose pointer member it does **not** own (just observes).

### The Rule of Three

> If you need to write **any one** of: destructor, copy constructor, or copy assignment operator — you almost certainly need to write **all three**.

The compiler-generated defaults are only correct for classes with no resource ownership. Once you add resource management (a destructor), the default copy behaviour becomes wrong.

## The = Operator Return Value

The assignment operator returns `ClassName &` (a reference to `*this`). This is necessary to support **chaining**:

```cpp
a = b = c;   // parsed as: a = (b = c)
```

If `operator=` returned `void`, the chain would fail. If it returned by value, it would invoke an extra copy. Returning a reference is efficient and correct.

In [ ]:
// Chaining assignment
StringBox x("X");
StringBox y("Y");
StringBox z("Z");

std::cout << "Before chain:" << std::endl;
x.print(); y.print(); z.print();

x = y = z;   // first z is copied to y, then y (now "Z") is copied to x

std::cout << "After  x = y = z:" << std::endl;
x.print(); y.print(); z.print();

## Final Exercise

**Exercise 3:** Implement the OCF for a class `Matrix` that represents a 2D integer matrix stored as a flat 1D array (`int *_data` of size `_rows * _cols`).

Requirements:
- A constructor `Matrix(int rows, int cols)` that allocates and zero-initialises the data.
- All four OCF functions (default constructor, copy constructor, copy assignment operator, destructor).
- `int  get(int row, int col) const`
- `void set(int row, int col, int val)`
- A `print()` method that prints the matrix row by row.

Test: create a 3x3 matrix, fill it, copy it, modify the original, and show that the copy is unchanged.

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### The Rule of Five

C++11 adds **move semantics**. When you can "steal" resources from a temporary instead of copying them, operations become much faster. This adds two more functions to the canonical set:

| Function | When Called |
|---|---|
| Move constructor `ClassName(ClassName &&other)` | Initialising from a temporary (rvalue) |
| Move assignment `ClassName &operator=(ClassName &&other)` | Assigning from a temporary |

### `= default` and `= delete`

```cpp
StringBox(const StringBox &) = delete;   // forbid copying entirely
~StringBox() = default;                  // explicitly use compiler-generated version
```

### `std::unique_ptr` reduces the need for OCF

Wrap owned pointers in `std::unique_ptr<char[]>` and the destructor, move constructor, and move assignment are handled automatically. Copying is disabled (which is usually what you want).

In [ ]:
#include <iostream>
#include <cstring>
#include <utility>   // std::move

class StringBoxModern {
public:
    StringBoxModern(const char *str = nullptr) : _content(nullptr), _length(0) {
        if (str) {
            _length  = strlen(str);
            _content = new char[_length + 1];
            strcpy(_content, str);
        }
    }

    // Move constructor -- steal resources from the temporary
    StringBoxModern(StringBoxModern &&other) noexcept
        : _content(other._content), _length(other._length) {
        std::cout << "[Move constructor] stealing data" << std::endl;
        other._content = nullptr;   // leave 'other' in a valid empty state
        other._length  = 0;
    }

    ~StringBoxModern() { delete[] _content; }

    void print() const {
        std::cout << (_content ? _content : "<empty>") << std::endl;
    }

private:
    char        *_content;
    unsigned int _length;
};

StringBoxModern src("Move me!");
std::cout << "src before move: "; src.print();

StringBoxModern dst(std::move(src));   // move constructor
std::cout << "dst after  move: "; dst.print();
std::cout << "src after  move: "; src.print();   // src is now empty